# 01 — Data Collection

Collects compound SMILES from the PubChem API, computes molecular descriptors with RDKit, and saves to `data/molecules.csv`.

## 1.1 Import Libraries

In [9]:
import os
import time
import requests
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

print("Libraries loaded successfully")

Libraries loaded successfully


## 1.2 Define Compound List

Defines 36 compounds across 3 classes.
- Class 0: Capsaicinoids
- Class 1: Vanilloids / Gingerols
- Class 2: Other Spice Alkaloids

In [10]:
# Compound dictionary: {compound_name: (class_id, class_name)}
compound_dict = {
    # Class 0 — Capsaicinoids (~12)
    "capsaicin":              (0, "Capsaicinoid"),
    "dihydrocapsaicin":       (0, "Capsaicinoid"),
    "nordihydrocapsaicin":    (0, "Capsaicinoid"),
    "homocapsaicin":          (0, "Capsaicinoid"),
    "homodihydrocapsaicin":   (0, "Capsaicinoid"),
    "nonivamide":             (0, "Capsaicinoid"),
    "norcapsaicin":           (0, "Capsaicinoid"),
    "dehydrocapsaicin":       (0, "Capsaicinoid"),
    "olvanil":                (0, "Capsaicinoid"),
    "nuvanil":                (0, "Capsaicinoid"),
    "civamide":               (0, "Capsaicinoid"),
    "resiniferatoxin":        (0, "Capsaicinoid"),

    # Class 1 — Vanilloids / Gingerols (~13)
    "6-gingerol":             (1, "Vanilloid_Gingerol"),
    "8-gingerol":             (1, "Vanilloid_Gingerol"),
    "10-gingerol":            (1, "Vanilloid_Gingerol"),
    "6-shogaol":              (1, "Vanilloid_Gingerol"),
    "8-shogaol":              (1, "Vanilloid_Gingerol"),
    "zingerone":              (1, "Vanilloid_Gingerol"),
    "paradol":                (1, "Vanilloid_Gingerol"),
    "vanillin":               (1, "Vanilloid_Gingerol"),
    "ferulic acid":           (1, "Vanilloid_Gingerol"),
    "eugenol":                (1, "Vanilloid_Gingerol"),
    "isoeugenol":             (1, "Vanilloid_Gingerol"),
    "methyl eugenol":         (1, "Vanilloid_Gingerol"),

    # Class 2 — Other Spice Alkaloids (~12)
    "piperine":               (2, "Spice_Alkaloid"),
    "piperlongumine":         (2, "Spice_Alkaloid"),
    "chavicine":              (2, "Spice_Alkaloid"),
    "isochavicine":           (2, "Spice_Alkaloid"),
    "capsazepine":            (2, "Spice_Alkaloid"),
    "anandamide":             (2, "Spice_Alkaloid"),
    "nabilone":               (2, "Spice_Alkaloid"),
    "capsiate":               (2, "Spice_Alkaloid"),
    "dihydrocapsiate":        (2, "Spice_Alkaloid"),
    "nordihydrocapsiate":     (2, "Spice_Alkaloid"),
    "vanillylamine":          (2, "Spice_Alkaloid"),
    "veratrylamine":          (2, "Spice_Alkaloid"),
}

print(f"Total compounds: {len(compound_dict)}")
for class_id in range(3):
    count = sum(1 for v in compound_dict.values() if v[0] == class_id)
    class_name = [v[1] for v in compound_dict.values() if v[0] == class_id][0]
    print(f"  Class {class_id} ({class_name}): {count}")

Total compounds: 36
  Class 0 (Capsaicinoid): 12
  Class 1 (Vanilloid_Gingerol): 12
  Class 2 (Spice_Alkaloid): 12


## 1.3 Fetch SMILES via PubChem API

Uses the PubChem PUG REST API to retrieve the IsomericSMILES for each compound.

In [11]:
PUBCHEM_BASE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name"

def fetch_smiles(compound_name):
    """Fetch IsomericSMILES for a compound from PubChem API."""
    url = f"{PUBCHEM_BASE_URL}/{compound_name}/property/IsomericSMILES/JSON"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        props = data["PropertyTable"]["Properties"][0]
        smiles = props.get("IsomericSMILES") or props.get("SMILES")
        if smiles is None:
            print(f"  [No key] {compound_name}: available keys = {list(props.keys())}")
        return smiles
    except requests.exceptions.HTTPError as e:
        print(f"  [HTTP error] {compound_name}: {e}")
        return None
    except Exception as e:
        print(f"  [Error] {compound_name}: {e}")
        return None

# Fetch SMILES for all compounds
smiles_data = []  # (compound_name, class_id, class_name, SMILES)

for compound_name, (class_label, class_name) in compound_dict.items():
    smiles = fetch_smiles(compound_name)
    if smiles is not None:
        smiles_data.append((compound_name, class_label, class_name, smiles))
        print(f"  [OK] {compound_name}")
    else:
        print(f"  [FAIL] {compound_name} — skipped")
    time.sleep(0.3)  # Rate limit delay

print(f"\nSMILES collection complete: {len(smiles_data)}/{len(compound_dict)} succeeded")

  [OK] capsaicin
  [OK] dihydrocapsaicin
  [OK] nordihydrocapsaicin
  [OK] homocapsaicin
  [OK] homodihydrocapsaicin
  [OK] nonivamide
  [OK] norcapsaicin
  [OK] dehydrocapsaicin
  [OK] olvanil
  [OK] nuvanil
  [OK] civamide
  [OK] resiniferatoxin
  [OK] 6-gingerol
  [OK] 8-gingerol
  [OK] 10-gingerol
  [OK] 6-shogaol
  [OK] 8-shogaol
  [OK] zingerone
  [OK] paradol
  [OK] vanillin
  [OK] ferulic acid
  [OK] eugenol
  [OK] isoeugenol
  [OK] methyl eugenol
  [OK] piperine
  [OK] piperlongumine
  [OK] chavicine
  [OK] isochavicine
  [OK] capsazepine
  [OK] anandamide
  [OK] nabilone
  [OK] capsiate
  [OK] dihydrocapsiate
  [OK] nordihydrocapsiate
  [OK] vanillylamine
  [OK] veratrylamine

SMILES collection complete: 36/36 succeeded


## 1.4 Convert SMILES to RDKit Mol Objects

In [12]:
# Convert SMILES to RDKit Mol objects
valid_molecules = []  # (compound_name, class_id, class_name, Mol object)

for compound_name, class_label, class_name, smiles in smiles_data:
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        valid_molecules.append((compound_name, class_label, class_name, mol))
    else:
        print(f"  [Parse failed] {compound_name}: {smiles}")

print(f"Valid molecules: {len(valid_molecules)}/{len(smiles_data)}")

Valid molecules: 36/36


## 1.5 Compute RDKit Molecular Descriptors

Computes all available 2D molecular descriptors (~200) provided by RDKit.

In [13]:
# Get all available descriptor names
descriptor_names = [desc[0] for desc in Descriptors.descList]
print(f"Available descriptors: {len(descriptor_names)}")

# Create descriptor calculator
descriptor_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# Compute descriptors for each molecule
compound_names_list = []
class_labels_list = []
class_names_list = []
descriptor_matrix = []

for compound_name, class_label, class_name, mol in valid_molecules:
    descriptors = descriptor_calculator.CalcDescriptors(mol)
    descriptor_matrix.append(descriptors)
    compound_names_list.append(compound_name)
    class_labels_list.append(class_label)
    class_names_list.append(class_name)

descriptor_matrix = np.array(descriptor_matrix, dtype=np.float64)
print(f"Descriptor matrix shape: {descriptor_matrix.shape}")

Available descriptors: 217
Descriptor matrix shape: (36, 217)


## 1.6 Build DataFrame & Preprocessing

Drops columns with excessive NaN values and zero-variance columns.

In [14]:
# Build DataFrame
molecule_df = pd.DataFrame(
    descriptor_matrix,
    columns=descriptor_names
)

# Add metadata columns
molecule_df.insert(0, "compound_name", compound_names_list)
molecule_df.insert(1, "class_label", class_labels_list)
molecule_df.insert(2, "class_name", class_names_list)

print(f"Shape before preprocessing: {molecule_df.shape}")
print(f"Columns with NaN: {molecule_df.isnull().any().sum()}")

Shape before preprocessing: (36, 220)
Columns with NaN: 0


In [15]:
# Select feature columns only (exclude metadata)
feature_columns = molecule_df.columns[3:]

# 1) Drop columns with >10% NaN
nan_threshold = 0.1 * len(molecule_df)
nan_counts = molecule_df[feature_columns].isnull().sum()
columns_to_drop_nan = nan_counts[nan_counts > nan_threshold].index.tolist()
print(f"Columns with >10% NaN: {len(columns_to_drop_nan)}")

molecule_df = molecule_df.drop(columns=columns_to_drop_nan)

# Fill remaining NaN with 0
feature_columns = molecule_df.columns[3:]
molecule_df[feature_columns] = molecule_df[feature_columns].fillna(0)

# 2) Drop zero-variance columns (constant features)
feature_variances = molecule_df[feature_columns].var()
zero_variance_columns = feature_variances[feature_variances == 0].index.tolist()
print(f"Zero-variance columns: {len(zero_variance_columns)}")

molecule_df = molecule_df.drop(columns=zero_variance_columns)

# Handle inf values
feature_columns = molecule_df.columns[3:]
molecule_df[feature_columns] = molecule_df[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)
molecule_df[feature_columns] = molecule_df[feature_columns].fillna(0)

print(f"\nShape after preprocessing: {molecule_df.shape}")
print(f"Number of features: {len(molecule_df.columns) - 3}")

Columns with >10% NaN: 0
Zero-variance columns: 65

Shape after preprocessing: (36, 155)
Number of features: 152


## 1.7 Save to CSV

In [16]:
output_path = os.path.join("data", "molecules.csv")
molecule_df.to_csv(output_path, index=False)
print(f"CSV saved: {output_path}")
print(f"Final shape: {molecule_df.shape}")

CSV saved: ../data/molecules.csv
Final shape: (36, 155)


## 1.8 Final Verification

Checks the class distribution and basic statistics of the dataset.

In [17]:
print("---- Class Distribution --------------------")
class_distribution = molecule_df.groupby(["class_label", "class_name"]).size()
print(class_distribution)

print("\n---- Data Summary ----------------------------")
print(f"Total compounds: {len(molecule_df)}")
print(f"Total features: {len(molecule_df.columns) - 3}")
print(f"\nFirst 5 compounds:")
print(molecule_df[["compound_name", "class_label", "class_name"]].head())

print("\n---- Feature Statistics (first 5) ------------")
print(molecule_df[molecule_df.columns[3:8]].describe())

=== Class Distribution ===
class_label  class_name        
0            Capsaicinoid          12
1            Vanilloid_Gingerol    12
2            Spice_Alkaloid        12
dtype: int64

=== Data Summary ===
Total compounds: 36
Total features: 152

First 5 compounds:
          compound_name  class_label    class_name
0             capsaicin            0  Capsaicinoid
1      dihydrocapsaicin            0  Capsaicinoid
2   nordihydrocapsaicin            0  Capsaicinoid
3         homocapsaicin            0  Capsaicinoid
4  homodihydrocapsaicin            0  Capsaicinoid

=== Feature Statistics (first 5) ===
       MaxAbsEStateIndex  MaxEStateIndex  MinAbsEStateIndex  MinEStateIndex  \
count          36.000000       36.000000          36.000000       36.000000   
mean           11.121948       11.121948           0.120054       -0.067517   
std             1.710434        1.710434           0.140874        0.425277   
min             5.163908        5.163908           0.002891       -1.801